<a href="https://colab.research.google.com/github/jespimentel/aula_fucape/blob/main/transcricao_ffmpeg_whisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Palestra: "O Poder de IA e do Python na Coleta e Tratamento de Dados Judiciais"
## FFmpeg e Whisper
---

Notebook de referência da palestra de **José Eduardo de Souza Pimentel** na FUCAPE - Jun. 2026

---

**Fluxo:**

### Parte 1 - FFmpeg
1. Instalação do FFmpeg
2. Upload de um arquivo de áudio/vídeo
3. Operações com FFmpeg: alterar formato, converter para MP3, segmentar
4. Comandos extras: comprimir (reduzir bitrate) e cortar um trecho específico

### Parte 2 - Whisper
1. Instalação do Whisper
2. Carregamento do modelo
3. Transcrição do áudio com Whisper

> **Observação n. 1:** para dados sigilosos (ex. depoimentos, audiências), o Whisper rodando aqui no Colab processa o áudio localmente na máquina virtual. É diferente, portanto, de enviar o arquivo a um chat de IA externo.
>
> **Observação n. 2:** transcrições automatizadas exigem revisão humana para serem incorporadas a documentos oficiais.

## Parte 1 — FFmpeg: Manipulação de Áudio/Vídeo

### Instalar o FFmpeg

In [1]:
!apt -qq update
!apt -qq install -y ffmpeg
!ffmpeg -version


43 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 43 not upgraded.
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-lib

### Upload do arquivo de áudio ou vídeo

In [2]:
from google.colab import files
import os

uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
nome_base, extensao_original = os.path.splitext(input_filename)

print(f"Arquivo carregado: {input_filename}")


Saving WhatsApp Video 2026-06-20 at 10.10.28.mp4 to WhatsApp Video 2026-06-20 at 10.10.28.mp4
Arquivo carregado: WhatsApp Video 2026-06-20 at 10.10.28.mp4


### 1. Alteração do formato de vídeo

Converte o vídeo carregado para outro contêiner/formato (ex.: mp4 → avi). Basta trocar `output_format`.

In [3]:
output_format = "avi"  # altere para o formato desejado: mp4, avi, mkv, mov, webm...
output_filename = f"{nome_base}_convertido.{output_format}"

!ffmpeg -y -i "{input_filename}" "{output_filename}"

print(f"Arquivo convertido: {output_filename}")


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### 2. Conversão para MP3

Extrai apenas o áudio do arquivo e salva como MP3 (`-vn` descarta o vídeo).

In [4]:
mp3_filename = f"{nome_base}.mp3"

!ffmpeg -y -i "{input_filename}" -vn -acodec libmp3lame -q:a 2 "{mp3_filename}"

print(f"Áudio extraído: {mp3_filename}")


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### 3. Segmentação (divisão em partes)

Divide o arquivo em blocos de duração fixa (`segment_time`, em segundos). Útil para respeitar limites de tamanho/duração de ferramentas de transcrição.

Observação: com `-c copy` o corte ocorre no keyframe mais próximo, então a duração de cada parte pode variar levemente. Para cortes exatos, remova `-c copy` (reencoda, processo mais lento).

In [5]:
segment_time = 60  # duração de cada segmento, em segundos

extensao = extensao_original if extensao_original else ".mp4"
padrao_saida = f"{nome_base}_parte_%03d{extensao}"

!ffmpeg -y -i "{input_filename}" -f segment -segment_time {segment_time} -reset_timestamps 1 -c copy "{padrao_saida}"

import glob
partes = sorted(glob.glob(f"{nome_base}_parte_*"))
print(f"{len(partes)} segmento(s) gerado(s):")
for p in partes:
    print(" -", p)


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### 4. Comandos extras
### 4.1 Compressão / redução de bitrate

Reduz o tamanho do arquivo de áudio diminuindo o bitrate. Útil quando a ferramenta de IA tem limite de upload (ex.: chats de IA costumam limitar o tamanho de arquivos enviados).

In [6]:
bitrate_alvo = "64k"  # exemplos: 64k, 96k, 128k (quanto menor, menor o arquivo e a qualidade)
comprimido_filename = f"{nome_base}_comprimido.mp3"

!ffmpeg -y -i "{input_filename}" -vn -b:a {bitrate_alvo} "{comprimido_filename}"

print(f"Arquivo comprimido: {comprimido_filename}")
!ls -lh "{input_filename}" "{comprimido_filename}"


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

### 4.2. Corte de um trecho específico

Extrai apenas um trecho do arquivo a partir de um ponto inicial (`inicio`) e por uma duração (`duracao`). Útil para isolar a fala de uma testemunha específica, por exemplo.

In [7]:
inicio = "00:00:30"   # timestamp de início (HH:MM:SS)
duracao = "00:01:00"  # duração do trecho a extrair (HH:MM:SS)

extensao = extensao_original if extensao_original else ".mp4"
trecho_filename = f"{nome_base}_trecho{extensao}"

!ffmpeg -y -i "{input_filename}" -ss {inicio} -t {duracao} -c copy "{trecho_filename}"

print(f"Trecho extraído: {trecho_filename}")


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

## Parte 2 — Transcrição com Whisper

### Instalação

O Whisper é um modelo de transcrição de código aberto. Roda localmente na máquina virtual do Colab (sem enviar o áudio a um serviço externo de IA).

In [8]:
!pip install -q -U openai-whisper


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.3 MB/s eta 0:00:00


### Carregar o modelo

Tamanhos disponíveis (do mais rápido/impreciso ao mais lento/preciso): `tiny`, `base`, `small`, `medium`, `large`.
Para português, `small` ou `medium` costumam dar um bom equilíbrio entre velocidade e qualidade numa demonstração ao vivo.

In [9]:
import whisper

modelo = whisper.load_model("small")


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 166MiB/s]


### Transcrever o áudio

In [10]:
audio_para_transcrever = mp3_filename  # pode trocar por comprimido_filename, trecho_filename, etc.

resultado = modelo.transcribe(audio_para_transcrever, language="pt")

print(resultado["text"])


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


 3 curiosidades sobre futebol. Sabe por que a bola é branca? Não. Por causa do São Paulo Futebol Clube. Não, está zoando. Conta é o seguinte, antigamente a bola era de cor, era marrom, escura. E aí quando acontecia os treinamentos, a bola sempre ia pro mato. Isso eu estou falando da época de 1920, 1923. E aí pro pegador de bola, era horrível pra ele achar a bola. Perdi muita bola, então ele teve uma ideia. Joaquim Simão, que era um funcionário do São Paulo. Ele falou, vou pintar a bola de branco, que aí vai ficar mais fácil pra eu achar. Aí começaram a pintar a bola de branco, acharam a ideia sensacional. Com isso conseguiu fazer os jogos à noite, porque era muito difícil, não tinha iluminação à noite. Então com a bola branca resolveu o problema, aí foi pro mundo inteiro, o mundo inteiro. Passou a usar a bola branca e é usada até hoje. Caramba, que legal! Segundo a curiosidade, por que o pegador de bola chama Gandula? É verdade, não faz nem sentido. Não faz sentido, né? É um nome esqui

### Transcrição com marcação de tempo por trecho

Formato mais próximo do necessário para uma ata ou registro de depoimento, com timestamps por fala.

In [11]:
nome_txt = f"{nome_base}_transcricao.txt"

with open(nome_txt, "w", encoding="utf-8") as f:
    for segmento in resultado["segments"]:
        inicio_s = segmento["start"]
        fim_s = segmento["end"]
        texto = segmento["text"].strip()
        linha = f"[{inicio_s:7.1f}s - {fim_s:7.1f}s] {texto}"
        print(linha)
        f.write(linha + "\n")

print(f"\nArquivo salvo: {nome_txt}")


[    0.0s -     3.0s] 3 curiosidades sobre futebol. Sabe por que a bola é branca?
[    3.0s -     3.5s] Não.
[    3.5s -     5.5s] Por causa do São Paulo Futebol Clube.
[    5.5s -     7.0s] Não, está zoando.
[    7.0s -    11.0s] Conta é o seguinte, antigamente a bola era de cor, era marrom, escura.
[   11.0s -    14.0s] E aí quando acontecia os treinamentos, a bola sempre ia pro mato.
[   14.0s -    17.0s] Isso eu estou falando da época de 1920, 1923.
[   17.0s -    21.0s] E aí pro pegador de bola, era horrível pra ele achar a bola.
[   21.0s -    23.0s] Perdi muita bola, então ele teve uma ideia.
[   23.0s -    26.0s] Joaquim Simão, que era um funcionário do São Paulo.
[   26.0s -    28.5s] Ele falou, vou pintar a bola de branco, que aí vai ficar mais fácil pra eu achar.
[   28.5s -    32.5s] Aí começaram a pintar a bola de branco, acharam a ideia sensacional.
[   32.5s -    37.5s] Com isso conseguiu fazer os jogos à noite, porque era muito difícil, não tinha iluminação à noite.
[  